In [ ]:
%matplotlib widget
# Boilerplate import code for all libraries
# Changes to the precision require re-loading the kernel and need to be done before any op uses them.
import warpSPHCore_config as swc
from typing import Any
swc.configure(precision="float32", dim=Any) # precision: float16|half|float32|single|float64|double

import warpSPHCore as sph
from warpSPHCore.type_config import *
print(get_type_config()) # confirms active settings

# Initialize warp at this point
import warp as wp; wp.init()

import os
import torch
if torch.cuda.is_available(): # set the TORCH_CUDA_ARCH_LIST environment variable to the compute capability of the GPU for faster compiles
    os.environ['TORCH_CUDA_ARCH_LIST'] = f'{torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}'

import warnings
from tqdm import TqdmExperimentalWarning
warnings.filterwarnings("ignore", category=TqdmExperimentalWarning)
from tqdm.autonotebook import tqdm

# final import blocks that are generic
import matplotlib.pyplot as plt
from torch.profiler import profile, record_function, ProfilerActivity
import numpy as np
import math
import shlex    
import subprocess
import shutil

# custom SPH libraries
from warpSPHIntegrators.integration import *
from warpSPHCore import *

# This library
from warpSPH import *
from warpSPH.modules.timestep.compressible import computeTimestep

# The case utilities that contain all the case setup functions for the various test cases
from warpSPH.caseUtils import *

# Sedov-Taylor Blastwave (2D)

This notebook runs the 2D Sedov-Taylor blastwave benchmark in the compressible SPH suite.

The case models radial blast expansion from a compact energy source and is used to assess isotropy, shock-front shape, and conservation in 2D.

This notebook follows the same reusable structure used across all 15 compressible benchmark cases:

1. Configure imports and numeric precision.
2. Define case-specific physical parameters and initial-condition data.
3. Build domain, solver, and scheme configuration from shared builders.
4. Sample and initialize particles/state for the selected case.
5. Run the time integration loop with diagnostics and adaptive timestep control.
6. Export trajectory/state snapshots and generate image frames during the run.
7. Finalize outputs by writing final state data and rendering media artifacts (for example MP4/GIF).

Precision note: switching between single and double precision is controlled in the import/configuration block. Because precision is set when core modules/kernels are initialized, any precision change requires a kernel restart before re-running the notebook.

![](outputs/07-Sedov_Taylor_Blastwave_2D.gif)

In [ ]:
nx = 200
dim = 2
L = 2
n_h = 4

goalRadius = 0.8
gamma = 5/3
rho0 = 1
E0 = 1

extraData = {
    'nx': nx,
    'dim': dim,
    'L': L,
    'n_h': n_h,

    'gamma': gamma,
    'rho0': rho0,
    'goalRadius': goalRadius,
}

In [ ]:
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
dtype = get_torch_precision()

config, integrator = buildConfig(
    domain = buildDomainDescription(L, dim, True, device, dtype),
    dim = dim,
    kernel = KernelFunctions.B7,
    targetNeighbors = n_h_to_nH(4, dim),
    supportMode = SupportScheme.KernelMeanSymmetric,
    gradientMode = GradientScheme.Difference,
    laplacianMode = LaplacianScheme.Brookshaw,
    integrationScheme = IntegrationSchemeType.rungeKutta2,
    samplingScheme = SamplingScheme.regular,
    device = device,
    dtype = dtype,
    dt = None,
    adaptiveDt = True,
    cflFactor=0.3,
)
config.nx = nx

config.minDt = 1e-8
# config.dx = L / (nx * 2)

scheme = CompressibleSPHScheme.CRKSPH
bundle = buildScheme(scheme)
SimulationSystem, SimulationState = bundle.SimulationSystem, bundle.SimulationState
SimulationUpdate = bundle.SimulationUpdate
fn, export_fn, import_fn = bundle.stepFunction, bundle.exportFunction, bundle.importFunction


schemeConfig = bundle.SimulationConfig()
schemeConfig.gamma = gamma
schemeConfig.rho0 = rho0


schemeConfig.viscositySwitchParams.scheme = ViscositySwitch.NoneSwitch
schemeConfig.adaptiveSupportScheme = AdaptiveSupportScheme.Owen
schemeConfig.adaptiveSupportCorrections = False


In [ ]:
from warpSPH.caseUtils.sedov import *

In [ ]:
compressibleSystem = buildSedov(
    SimulationSystem, SimulationState, 
    config = config,
    nx = nx, dim = dim, domainExtent = L, 
    periodicDomain = True, 
    rho0 = rho0, E0 = E0, 
    initialization = 'hat', 
    gamma = gamma, kernel = config.kernel, 
    targetNeighbors = config.targetNeighbors, 
    dtype = config.dtype, device = config.device)

In [ ]:
from warpSPH.caseUtils.sedov.sedovSolution import SedovSolution, radius, beta
from scipy.optimize import minimize

answer = SedovSolution(
    nDim = dim,
    gamma = gamma,
    rho0 = rho0,
    E0 = E0,
    h0 = 2/nx
)
nu1 = 1.0/(answer.nu + 2.0)
nu2 = 2.0*nu1
goalTime = (goalRadius*(answer.alpha*rho0/E0)**nu1)**(1.0/nu2)
vs, r2, v2, rho2, P2 = answer.shockState(goalTime)
rad_t = lambda t: radius(beta(dim), E0, t, rho0, 1)

targetTime = minimize(lambda t: (rad_t(t) - goalRadius) ** 2, 0.2).x[0]

print(f'Target Time [paper]: {targetTime:8.4g}, {rad_t(targetTime):8.4g} / {answer.shockState(targetTime)[1]:8.4g}')
print(f'Target Time  [code]: {goalTime:8.4g}, {rad_t(goalTime):8.4g} / {answer.shockState(goalTime)[1]:8.4g}')

In [ ]:
from warpSPHPlotting import visualize, PlottingOptions, PlotScaling, GridVisualization, UniformColorMap
markerSize = 0.5
plotter = visualize(
    particleState = compressibleSystem.state,
    domain = config.domain,
    quantities = {
        "A": compressibleSystem.state.internalEnergies,
        "B": compressibleSystem.state.densities,
    },
    plotOptions = {
        "A": PlottingOptions(
            colorMap = UniformColorMap.viridis,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Logarithmic,
            plotTitle = "internal energy",
            gridVisualization = GridVisualization(
                resolution = 1024,
            ),
            vMin=1e-10
        ),
        "B": PlottingOptions(
            colorMap = UniformColorMap.cividis,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
            plotTitle = "density",
            # gridVisualization = GridVisualization(
            #     resolution = 1024,
            # ),
        ),
    },
    figTitle = "Wave Equation Example",
    mosaic = 'AB',
    figsize= (11,5),
    backend='vispy',
    # backend='pyVista',
    # backendOptions = {
    #     # In notebooks, use trame for reliable live updates.
    #     'jupyter_backend': 'trame',
    # }
)

# if args.exportImages:
#     plotter.export(f'output/{folderName}/frame_00000.png', dpi = args.figureDpi)

In [ ]:
def plotShock(fig, axis, runningState):
    KE = (runningState.state.masses * torch.linalg.norm(runningState.state.velocities, dim = -1)**2 / 2).sum()
    IE = (runningState.state.masses * runningState.state.internalEnergies).sum()
    TE = (IE + KE)

    t = runningState.t.cpu().item() if isinstance(runningState.t, torch.Tensor) else runningState.t
    if t == 0.0:
        t += 1e-8
    fig.suptitle(f'Time: {t:.4f} s - KE: {KE:.4g}, IE: {IE:.4g}, TE: {TE:.4g}')

    r = torch.linalg.norm(runningState.state.positions, dim = -1).cpu()
    theta = torch.atan2(runningState.state.positions[:,1], runningState.state.positions[:,0]).cpu()

    sc = axis[0,0].scatter(r, runningState.state.densities.cpu(), c = theta, s = markerSize, cmap = 'twilight')
    axis[0,0].set_xlabel('Radius')
    # axis[0,0].set_ylabel('Density')
    axis[0,0].grid(True)
    axis[0,0].set_axisbelow(True)
    axis[0,0].set_title('Density')
    axis[0,0].set_xlim(0, 1)

    sc = axis[0,1].scatter(r, runningState.state.internalEnergies.cpu(), c = theta, s = markerSize, cmap = 'twilight')
    axis[0,1].set_xlabel('Radius')
    # axis[0,1].set_ylabel('Internal Energy')
    axis[0,1].grid(True)
    axis[0,1].set_axisbelow(True)
    axis[0,1].set_yscale('log')
    axis[0,1].set_title('Internal Energy')
    axis[0,1].set_xlim(0, 1)

    sc = axis[1,0].scatter(r, torch.linalg.norm(runningState.state.velocities.cpu(), dim = -1), c = theta, s = markerSize, cmap = 'twilight')
    axis[1,0].set_xlabel('Radius')
    # axis[1,0].set_ylabel('Velocity')
    axis[1,0].grid(True)
    axis[1,0].set_axisbelow(True)
    axis[1,0].set_title('Velocity')
    axis[1,0].set_xlim(0, 1)

    sc = axis[1,1].scatter(r, runningState.state.supports.cpu(), c = theta, s = markerSize, cmap = 'twilight')
    axis[1,1].set_xlabel('Radius')
    # axis[1,1].set_ylabel('Support')
    axis[1,1].grid(True)
    axis[1,1].set_axisbelow(True)
    axis[1,1].set_title('Support')
    axis[1,1].set_xlim(0, 1)

    def radius(beta, E0, t, rho0, nu):
        return beta *( E0 * t**2 / (rho0)) ** (1/(2+nu))
    def velocity(beta, E0, t, rho0, nu):
        return 2/(nu+2) * radius(beta, E0, t, rho0, nu) / t

    def beta(nu):
        if nu == 1:
            # return 1 / (answer.alpha ** nu1 )
            return 1.11
        elif nu == 2:
            return 1.12
        elif nu == 3:
            return 1.15
        
    r = radius(beta(dim), E0, t, rho0, 2)
    v = velocity(beta(dim), E0, t, rho0, 2)
    # print(r)

    answer = SedovSolution(
        nDim = dim,
        gamma = gamma,
        rho0 = rho0,
        E0 = E0,
        h0 = 2/nx
    )
    vs, r2, v2, rho2, P2 = answer.shockState(t)
    axis[0,0].axvline(r2, color = 'red', linestyle = '--', label = 'Analytical Shock Position')
    axis[0,0].axhline(rho2, color = 'green', linestyle = '--', label = 'Analytical Shock Density')
    axis[0,1].axvline(r2, color = 'red', linestyle = '--', label = 'Analytical Shock Position')
    axis[1,0].axvline(r2, color = 'red', linestyle = '--', label = 'Analytical Shock Position')
    axis[1,0].axhline(v2, color = 'green', linestyle = '--', label = 'Analytical Shock Velocity')

    rhos = rho0 * (1+gamma)/(gamma-1)
    axis[0,0].axvline(r, color = 'orange', linestyle = '--', label = 'Estimated Shock Position')
    axis[0,0].axhline(rhos, color = 'purple', linestyle = '--', label = 'Estimated Shock Density')
    axis[0,1].axvline(r, color = 'orange', linestyle = '--', label = 'Estimated Shock Position')
    axis[1,0].axvline(r, color = 'orange', linestyle = '--', label = 'Estimated Shock Position')
    # axis[1,0].axhline(v, color = 'purple', linestyle = '--', label = 'Estimated Shock Velocity')

    # fig.colorbar = plt.colorbar(sc, ax = axis[0,0])

In [ ]:
runningState = compressibleSystem.initializeNewState()

kineticEnergy = 0.5 * (torch.linalg.norm(runningState.state.velocities, dim = -1) **2 * runningState.state.masses).sum()
thermalEnergy = (runningState.state.internalEnergies * runningState.state.masses).sum()
totalEnergy = kineticEnergy + thermalEnergy

In [ ]:
caseName = '07-Sedov_Taylor_Blastwave_2D'
exportPath = prepExport(f'{caseName}', config, schemeConfig, scheme, export_fn)
exportSimulationSystem(exportPath, 'initialState', scheme, compressibleSystem, exportAdjacency = False, stages = None, exportStagesAdjacency = False, extraData = dict({
    'kineticEnergy': kineticEnergy,
    'thermalEnergy': thermalEnergy,
    'totalEnergy': totalEnergy,
    'frame_num': 0,
}, **extraData))


In [ ]:
fig, axis = plt.subplots(2,2, figsize = (9, 5), squeeze = False)

plotShock(fig, axis, runningState)

fig.tight_layout()

imagePath = f'{exportPath}/images'
os.makedirs(imagePath, exist_ok = True)
fig.savefig(f'{imagePath}/frame_{0:05d}.png')


In [ ]:
# config.dt = 1e-4
t_limit = goalTime
nSteps = int(t_limit / config.dt)

print(f"Running with dt: {config.dt}, which gives nSteps: {nSteps}")
# nSteps = 20

runningState = compressibleSystem.initializeNewState()

trajectory = []

priorStep = None
for i in (tq := tqdm(range(nSteps), leave = True)):
    begin = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    begin.record()
    result = integrator.function(
        state = runningState,
        f = fn,
        dt = config.dt,  
        config = config,
        schemeConfig = schemeConfig,
        verbose = False,
        # priorStep = priorStep
    )
    end.record()
    torch.cuda.synchronize()
    priorStep = result.stages[-1]
    timing = begin.elapsed_time(end)

    runningState = result.state
    kineticEnergy = 0.5 * (torch.linalg.norm(runningState.state.velocities, dim = -1) **2 * runningState.state.masses).sum()
    thermalEnergy = (runningState.state.internalEnergies * runningState.state.masses).sum()
    totalEnergy = kineticEnergy + thermalEnergy

    trajectory.append(
        (i, (i+1)*config.dt, totalEnergy.item(), kineticEnergy.item(), thermalEnergy.item(), timing)
,     )

    tq.set_description(f"Step {i+1}/{nSteps}, time: {(i+1)*config.dt:8.4g}/{t_limit:8.4g}, TE: {totalEnergy:.3g}, KE: {kineticEnergy:.3g}, IE: {thermalEnergy:.3g}")
    # t = {runningState.t:2f}, dt = {config.dt:.3g}, ptcls = {len(runningState.state.positions)}\nTotal Energy: {totalEnergy:.3g}, Kinetic Energy: {kineticEnergy:.3g}, Thermal Energy: {thermalEnergy:.3g}'
    # break
    if i % 25 == 0:
        axis[0,0].cla()
        axis[0,1].cla()
        axis[1,0].cla()
        axis[1,1].cla()
        plotShock(fig, axis, runningState)
        # fig.tight_layout()
        fig.canvas.draw()
        fig.canvas.flush_events()
        fig.savefig(f'{imagePath}/frame_{i:05d}.png')
        
    if i % 500 == 0:
        exportSimulationSystem(exportPath, f'state_{i:04d}', scheme, runningState, exportAdjacency = False, stages = result.stages, exportStagesAdjacency = True, extraData = dict(**extraData, **{
            'kineticEnergy': kineticEnergy,
            'thermalEnergy': thermalEnergy,
            'totalEnergy': totalEnergy,
            'frame_num': i,
        }))
        

In [ ]:
exportSimulationSystem(exportPath, f'finalState', scheme, runningState, exportAdjacency = False, stages = result.stages, exportStagesAdjacency = True, extraData = dict(**extraData, **{
    'kineticEnergy': kineticEnergy,
    'thermalEnergy': thermalEnergy,
    'totalEnergy': totalEnergy,
    'frame_num': i,
}))

In [ ]:
ffmpeg_cmd = "ffmpeg -y -loglevel error -hide_banner -framerate 50 -f image2 -pattern_type glob -i 'frame_*.png' -c:v libx264 -pix_fmt yuv420p -b:v 10M output.mp4"
subprocess.run(shlex.split(ffmpeg_cmd), check=True, cwd = imagePath)
ffmpeg_cmd = 'ffmpeg -y -loglevel error -hide_banner -i output.mp4  -vf "fps=50,scale=540:-1:flags=lanczos,palettegen" palette.png'
subprocess.run(shlex.split(ffmpeg_cmd), check=True, cwd = imagePath)
ffmpeg_cmd = 'ffmpeg -y -loglevel error -hide_banner -i output.mp4 -i palette.png -filter_complex "fps=25,scale=540:-1:flags=lanczos[x];[x][1:v]paletteuse" out.gif'
subprocess.run(shlex.split(ffmpeg_cmd), check=True, cwd = imagePath)

# now copy the output.mp4 and out.gif to the parent directory for easier access
shutil.copy(f'{imagePath}/output.mp4', f'{exportPath}/output.mp4')
shutil.copy(f'{imagePath}/out.gif', f'{exportPath}/out.gif');